In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

category_df = spark.table(
    f"{catalog}.ecommerce_bronze.category_translation"
)

category_df.printSchema()
category_df.show(10, truncate=False)

In [0]:
category_df.select([
    F.count(F.when(category_df[c].isNull(), 1)).alias(c)
    for c in category_df.columns
]).show()

In [0]:
silver_category = (
    category_df
    .dropDuplicates(["product_category_name"])
    .withColumn(
        "product_category_name",
        F.trim(category_df["product_category_name"])
    )
    .withColumn(
        "product_category_name_english",
        F.trim(category_df["product_category_name_english"])
    )
)


In [0]:
silver_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/category_translation"

silver_category.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_silver.category_translation
USING DELTA
LOCATION '{silver_path}'
""")

In [0]:
spark.table(
    f"{catalog}.ecommerce_silver.category_translation"
).show(10, truncate=False)